# Software-as-a-Graph (SaG) — JSS Journal Paper Reproducibility Suite
### *Heterogeneous Graph Learning for Pre-Deployment Reliability and Dependability Analysis of Complex Distributed Systems*
**Journal of Systems and Software (JSS) — Special Issue VSI:AI4MSS**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onuralpyigit/SoftwareAsAGraph/blob/main/notebooks/train_gnn_colab.ipynb)

This notebook provides a cloud-ready, GPU-accelerated environment to train all GNN models and reproduce the empirical results, tables, and figures for the JSS submission.

---

### Simulator & Stage Decoupling Architecture

The SaG framework strictly decouples simulation engines across four architectural stages:

| Stage | Primary Engine | Metric / Role | Purpose & Criteria |
|---|---|---|---|
| **Predict** | `FaultInjector` | $I^*(v)$ (Feed-Loss Fraction) | High-throughput topological oracle generating supervised ground-truth labels for GNN training ($Q^*(v)$). |
| **Validate (Tier-1)** | `FaultInjector` | Ranking ($
ho \ge 0.70$), Top-$K$ Identification ($F_1@K$) | **Primary Static & Structural Validation Gate / Core Blocking Gate**: Evaluates GNN rank correlation and critical set identification; blocks non-viable models. |
| **Validate (Tier-2)** | `MessageFlowSimulator` | $I_{\text{dyn}}(v)$ (Traffic Delivery Degradation) | **Dynamic Behavioral Gate / Targeted Behavioral Gate**: High-fidelity discrete-event traffic simulation ($\rho_{\text{util}} = 0.65$, `qos_mode=full`) on candidate Top-$K$ components; also serves as a convergent-validity probe. |
| **Simulate / Explain** | `FailureSimulator`, `ChangePropagationSimulator` | Multi-dimensional $I_{\text{comp}}(v)$ and $IR, IM, IA, IS$ | **Explanatory Layer (ISO/IEC 25010 Quality Gates)**: Decomposes failure impact into Reliability, Maintainability, Availability, and Safety for multi-criteria architectural trade-offs. |
| **Prescribe** | `FailureSimulator` (via `EditVerifier`) | Net Impact Reduction ($\Delta I_{\text{comp}} > 0, \Delta \text{SRI} > 0$) | **Remediation Verifier**: Counterfactual simulation testing whether proposed refactoring edits genuinely mitigate risk without secondary cascade regressions. |

---

### Predictor Family Taxonomy & Dual-Engine Protocol

The SaG predictive pipeline evaluates four primary predictor configurations drawn from three families, alongside an operational dual-engine consensus protocol:

1. **Heterogeneous Graph Transformer (`HGT-QoS` / proposed)**: Relation-specific HGT message passing consuming the full native multigraph with 16-dimensional continuous-categorical edge features encoding middleware QoS contracts. Trained via multi-task dimension-masked loss with temperature-scaled ListMLE ranking ($\mathcal{L}_{\text{rank}}$ with $\tau = 1.0$) and pairwise margin ranking ($\mathcal{L}_{\text{pairwise}}$ with $\gamma = 0.05$).
2. **Homogeneous Graph Learning (`GAT`, `GAT-QoS`)**: Untyped message passing consuming scalar QoS aggregates.
3. **Training-Free Structural Baselines (`Topo`, `Topo-QoS`)**: Evaluated on the dynamically derived application flow projection ($G_{\text{flow}}$) to overcome pub-sub betweenness collapse. `Topo` combines betweenness ($0.6$) and articulation points ($0.4$). `Topo-QoS` weights shortest paths by inverse QoS severity $d = 1 / (w_{\text{qos}} + \epsilon)$.
4. **Dual-Engine Consensus Protocol (`DualEnginePredictor`)**: Operationalizes JSS Section 8.1 by running `HGT-QoS` and `Topo-QoS` concurrently. Partitions components into:
   - $S_{\text{consensus}} = \text{Top}_K(\text{HGT-QoS}) \cap \text{Top}_K(\text{Topo-QoS})$: Unanimous architectural risks gated automatically in CI/CD.
   - $S_{\text{diverge}} = \{v \in V_{\text{app}} \mid |\text{rank}_{\text{HGT-QoS}}(v) - \text{rank}_{\text{Topo-QoS}}(v)| > \Delta_{\text{thresh}}\}$: Escalated to human architectural review.

---

### Hardware Accelerator Setup (Colab GPU)
1. In the Colab menu, go to **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** (free tier) or **A100 / V100** (Colab Pro).
3. Click **Save**.

## 1. Workspace Setup
Clones the repository from GitHub for code + committed data (`data/scenarios/`). Two gitignored cache directories are required and can only be rebuilt against a live Neo4j instance, which Colab doesn't provide — so both are pulled in separately from Google Drive as pre-built tarballs instead of being part of the clone:
- `output/loso_cache/` — required by `main_table.py`/`loso_all_variants.py`/`kfold_all_variants.py`. Must contain all 12 paper scenarios (`atm_system`, `av_system`, `iot_smart_city_system`, `financial_trading_system`, `healthcare_system`, `hub_and_spoke_system`, `microservices_system`, `enterprise_system`, `telecom_ran_system`, `industrial_scada_system`, `realtime_gaming_system`, `logistics_fleet_system`) — LOSO (Table 8) and k-fold (Table 9) discover scenarios purely from what's physically present under it, with no flag to catch a partial cache.
- `output/realworld_cache/` — required by `realworld_zeroshot.py` (Table 11). Must contain all 5 real-world systems (`realworld_autoware_ros2`, `realworld_cloud_microservices`, `realworld_trainticket`, `realworld_homeassistant`, `realworld_edgex`). Kept as a **separate** directory from `loso_cache` — `discover_scenarios` treats every folder it finds as a LOSO fold, so merging the two would silently change the 12-fold corpus behind every published LOSO number.

The cell below checks coverage for both and warns if anything is missing.

**One-time local pre-flight** (run on a machine with both caches already built, e.g. via `make -f reproduce/Makefile cache` for the synthetic one and `scripts/populate_loso_cache.sh` pointed at `CACHE_DIR=output/realworld_cache` for the real-world one — see `reproduce/realworld_zeroshot.py`'s docstring):
```bash
tar -czf output/loso_cache.tar.gz -C output loso_cache
tar -czf output/realworld_cache.tar.gz -C output realworld_cache
```
Upload both to `My Drive/SaG/` once. The cell below then mounts Drive only to fetch those two files.

In [ ]:
#
 
-
-
-
 
P
r
i
m
a
r
y
 
p
a
t
h
:
 
c
l
o
n
e
 
c
o
d
e
 
f
r
o
m
 
G
i
t
H
u
b
,
 
p
u
l
l
 
l
o
s
o
_
c
a
c
h
e
 
f
r
o
m
 
D
r
i
v
e
 
-
-
-

i
m
p
o
r
t
 
o
s

f
r
o
m
 
p
a
t
h
l
i
b
 
i
m
p
o
r
t
 
P
a
t
h


R
E
P
O
_
D
I
R
 
=
 
"
/
c
o
n
t
e
n
t
/
S
o
f
t
w
a
r
e
A
s
A
G
r
a
p
h
"

i
f
 
n
o
t
 
o
s
.
p
a
t
h
.
e
x
i
s
t
s
(
R
E
P
O
_
D
I
R
)
:

 
 
 
 
!
g
i
t
 
c
l
o
n
e
 
h
t
t
p
s
:
/
/
g
i
t
h
u
b
.
c
o
m
/
o
n
u
r
a
l
p
y
i
g
i
t
/
S
o
f
t
w
a
r
e
A
s
A
G
r
a
p
h
.
g
i
t
 
{
R
E
P
O
_
D
I
R
}

e
l
s
e
:

 
 
 
 
#
 
P
u
l
l
 
l
a
t
e
s
t
 
c
h
a
n
g
e
s
 
i
f
 
t
h
e
 
r
e
p
o
 
a
l
r
e
a
d
y
 
e
x
i
s
t
s

 
 
 
 
!
c
d
 
{
R
E
P
O
_
D
I
R
}
 
&
&
 
g
i
t
 
p
u
l
l



%
c
d
 
{
R
E
P
O
_
D
I
R
}

%
s
e
t
_
e
n
v
 
P
Y
T
H
O
N
P
A
T
H
=
.


f
r
o
m
 
g
o
o
g
l
e
.
c
o
l
a
b
 
i
m
p
o
r
t
 
d
r
i
v
e

d
r
i
v
e
.
m
o
u
n
t
(
'
/
c
o
n
t
e
n
t
/
d
r
i
v
e
'
)


d
e
f
 
_
e
x
t
r
a
c
t
_
c
a
c
h
e
(
t
a
r
b
a
l
l
_
n
a
m
e
,
 
d
e
s
t
_
d
i
r
)
:

 
 
 
 
t
a
r
b
a
l
l
 
=
 
f
"
/
c
o
n
t
e
n
t
/
d
r
i
v
e
/
M
y
D
r
i
v
e
/
S
a
G
/
{
t
a
r
b
a
l
l
_
n
a
m
e
}
"

 
 
 
 
i
f
 
o
s
.
p
a
t
h
.
e
x
i
s
t
s
(
t
a
r
b
a
l
l
)
:

 
 
 
 
 
 
 
 
!
m
k
d
i
r
 
-
p
 
o
u
t
p
u
t

 
 
 
 
 
 
 
 
!
t
a
r
 
-
x
z
f
 
{
t
a
r
b
a
l
l
}
 
-
C
 
o
u
t
p
u
t
/

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
✓
 
{
d
e
s
t
_
d
i
r
}
 
e
x
t
r
a
c
t
e
d
 
f
r
o
m
 
D
r
i
v
e
.
"
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
⚠
️
 
{
t
a
r
b
a
l
l
}
 
n
o
t
 
f
o
u
n
d
 
—
 
c
e
l
l
s
 
d
e
p
e
n
d
i
n
g
 
o
n
 
{
d
e
s
t
_
d
i
r
}
 
w
i
l
l
 
f
a
i
l
 
"

 
 
 
 
 
 
 
 
 
 
 
 
 
 
"
u
n
t
i
l
 
i
t
'
s
 
b
u
i
l
t
 
l
o
c
a
l
l
y
 
a
n
d
 
u
p
l
o
a
d
e
d
 
t
h
e
r
e
.
"
)


_
e
x
t
r
a
c
t
_
c
a
c
h
e
(
"
l
o
s
o
_
c
a
c
h
e
.
t
a
r
.
g
z
"
,
 
"
o
u
t
p
u
t
/
l
o
s
o
_
c
a
c
h
e
"
)

_
e
x
t
r
a
c
t
_
c
a
c
h
e
(
"
r
e
a
l
w
o
r
l
d
_
c
a
c
h
e
.
t
a
r
.
g
z
"
,
 
"
o
u
t
p
u
t
/
r
e
a
l
w
o
r
l
d
_
c
a
c
h
e
"
)


#
 
-
-
-
 
S
c
e
n
a
r
i
o
 
c
o
v
e
r
a
g
e
 
c
h
e
c
k
 
-
-
-

#
 
l
o
s
o
_
a
l
l
_
v
a
r
i
a
n
t
s
.
p
y
 
/
 
k
f
o
l
d
_
a
l
l
_
v
a
r
i
a
n
t
s
.
p
y
 
t
a
k
e
 
n
o
 
-
-
s
c
e
n
a
r
i
o
s
 
f
l
a
g
:
 
t
h
e
y

#
 
d
i
s
c
o
v
e
r
 
s
c
e
n
a
r
i
o
s
 
p
u
r
e
l
y
 
f
r
o
m
 
w
h
a
t
e
v
e
r
 
f
o
l
d
e
r
s
 
a
r
e
 
p
h
y
s
i
c
a
l
l
y
 
p
r
e
s
e
n
t
 
u
n
d
e
r

#
 
-
-
c
a
c
h
e
-
d
i
r
.
 
A
 
t
a
r
b
a
l
l
 
b
u
i
l
t
 
b
e
f
o
r
e
 
a
 
d
o
m
a
i
n
 
w
a
s
 
a
d
d
e
d
 
(
o
r
 
b
u
i
l
t
 
w
i
t
h
 
o
n
l
y
 
t
h
e

#
 
7
 
i
n
-
d
i
s
t
r
i
b
u
t
i
o
n
 
s
c
e
n
a
r
i
o
s
)
 
w
i
l
l
 
s
i
l
e
n
t
l
y
 
r
u
n
 
a
n
 
8
-
 
o
r
 
7
-
f
o
l
d
 
L
O
S
O
 
i
n
s
t
e
a
d
 
o
f

#
 
t
h
e
 
p
a
p
e
r
'
s
 
1
2
-
f
o
l
d
 
L
O
S
O
 
(
J
S
S
 
T
a
b
l
e
 
8
)
,
 
w
i
t
h
 
n
o
 
e
r
r
o
r
 
a
n
y
w
h
e
r
e
 
d
o
w
n
s
t
r
e
a
m
 
—
 
s
o

#
 
c
h
e
c
k
 
c
o
v
e
r
a
g
e
 
h
e
r
e
,
 
b
e
f
o
r
e
 
s
p
e
n
d
i
n
g
 
G
P
U
 
t
i
m
e
 
o
n
 
a
 
r
u
n
 
t
h
a
t
 
u
n
d
e
r
-
c
o
v
e
r
s
 
i
t
.

R
E
Q
U
I
R
E
D
_
L
O
S
O
_
S
C
E
N
A
R
I
O
S
 
=
 
[

 
 
 
 
"
a
t
m
_
s
y
s
t
e
m
"
,
 
"
a
v
_
s
y
s
t
e
m
"
,
 
"
i
o
t
_
s
m
a
r
t
_
c
i
t
y
_
s
y
s
t
e
m
"
,
 
"
f
i
n
a
n
c
i
a
l
_
t
r
a
d
i
n
g
_
s
y
s
t
e
m
"
,

 
 
 
 
"
h
e
a
l
t
h
c
a
r
e
_
s
y
s
t
e
m
"
,
 
"
h
u
b
_
a
n
d
_
s
p
o
k
e
_
s
y
s
t
e
m
"
,
 
"
m
i
c
r
o
s
e
r
v
i
c
e
s
_
s
y
s
t
e
m
"
,

 
 
 
 
"
e
n
t
e
r
p
r
i
s
e
_
s
y
s
t
e
m
"
,
 
"
t
e
l
e
c
o
m
_
r
a
n
_
s
y
s
t
e
m
"
,
 
"
i
n
d
u
s
t
r
i
a
l
_
s
c
a
d
a
_
s
y
s
t
e
m
"
,

 
 
 
 
"
r
e
a
l
t
i
m
e
_
g
a
m
i
n
g
_
s
y
s
t
e
m
"
,
 
"
l
o
g
i
s
t
i
c
s
_
f
l
e
e
t
_
s
y
s
t
e
m
"
,

]

R
E
Q
U
I
R
E
D
_
R
E
A
L
W
O
R
L
D
_
S
Y
S
T
E
M
S
 
=
 
[

 
 
 
 
"
r
e
a
l
w
o
r
l
d
_
a
u
t
o
w
a
r
e
_
r
o
s
2
"
,
 
"
r
e
a
l
w
o
r
l
d
_
c
l
o
u
d
_
m
i
c
r
o
s
e
r
v
i
c
e
s
"
,

 
 
 
 
"
r
e
a
l
w
o
r
l
d
_
t
r
a
i
n
t
i
c
k
e
t
"
,
 
"
r
e
a
l
w
o
r
l
d
_
h
o
m
e
a
s
s
i
s
t
a
n
t
"
,
 
"
r
e
a
l
w
o
r
l
d
_
e
d
g
e
x
"
,

]


d
e
f
 
_
c
h
e
c
k
_
c
o
v
e
r
a
g
e
(
c
a
c
h
e
_
d
i
r
,
 
r
e
q
u
i
r
e
d
,
 
l
a
b
e
l
)
:

 
 
 
 
c
a
c
h
e
_
d
i
r
 
=
 
P
a
t
h
(
c
a
c
h
e
_
d
i
r
)

 
 
 
 
p
r
e
s
e
n
t
 
=
 
s
o
r
t
e
d
(
p
.
n
a
m
e
 
f
o
r
 
p
 
i
n
 
c
a
c
h
e
_
d
i
r
.
i
t
e
r
d
i
r
(
)
 
i
f
 
p
.
i
s
_
d
i
r
(
)
)
 
i
f
 
c
a
c
h
e
_
d
i
r
.
e
x
i
s
t
s
(
)
 
e
l
s
e
 
[
]

 
 
 
 
m
i
s
s
i
n
g
 
=
 
[
s
 
f
o
r
 
s
 
i
n
 
r
e
q
u
i
r
e
d
 
i
f
 
s
 
n
o
t
 
i
n
 
p
r
e
s
e
n
t
]

 
 
 
 
i
f
 
m
i
s
s
i
n
g
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
⚠
️
 
{
c
a
c
h
e
_
d
i
r
}
 
i
s
 
m
i
s
s
i
n
g
 
{
l
e
n
(
m
i
s
s
i
n
g
)
}
/
{
l
e
n
(
r
e
q
u
i
r
e
d
)
}
 
r
e
q
u
i
r
e
d
 
{
l
a
b
e
l
}
:
 
{
m
i
s
s
i
n
g
}
"
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
✓
 
A
l
l
 
{
l
e
n
(
r
e
q
u
i
r
e
d
)
}
 
{
l
a
b
e
l
}
 
p
r
e
s
e
n
t
 
i
n
 
{
c
a
c
h
e
_
d
i
r
}
:
 
{
p
r
e
s
e
n
t
}
"
)


_
c
h
e
c
k
_
c
o
v
e
r
a
g
e
(
"
o
u
t
p
u
t
/
l
o
s
o
_
c
a
c
h
e
"
,
 
R
E
Q
U
I
R
E
D
_
L
O
S
O
_
S
C
E
N
A
R
I
O
S
,
 
"
L
O
S
O
 
s
c
e
n
a
r
i
o
s
"
)

_
c
h
e
c
k
_
c
o
v
e
r
a
g
e
(
"
o
u
t
p
u
t
/
r
e
a
l
w
o
r
l
d
_
c
a
c
h
e
"
,
 
R
E
Q
U
I
R
E
D
_
R
E
A
L
W
O
R
L
D
_
S
Y
S
T
E
M
S
,
 
"
r
e
a
l
-
w
o
r
l
d
 
s
y
s
t
e
m
s
"
)


#
 
-
-
-
 
A
l
t
e
r
n
a
t
i
v
e
:
 
m
o
u
n
t
 
t
h
e
 
w
h
o
l
e
 
p
r
o
j
e
c
t
 
f
r
o
m
 
D
r
i
v
e
 
i
n
s
t
e
a
d
 
o
f
 
c
l
o
n
i
n
g
 
-
-
-

#
 
f
r
o
m
 
g
o
o
g
l
e
.
c
o
l
a
b
 
i
m
p
o
r
t
 
d
r
i
v
e

#
 
d
r
i
v
e
.
m
o
u
n
t
(
'
/
c
o
n
t
e
n
t
/
d
r
i
v
e
'
)

#
 
#
 
O
p
t
i
o
n
:
 
e
n
t
i
r
e
 
p
r
o
j
e
c
t
 
z
i
p
p
e
d
 
i
n
 
D
r
i
v
e
:

#
 
#
 
!
u
n
z
i
p
 
-
q
 
/
c
o
n
t
e
n
t
/
d
r
i
v
e
/
M
y
D
r
i
v
e
/
S
o
f
t
w
a
r
e
A
s
A
G
r
a
p
h
.
z
i
p
 
-
d
 
/
c
o
n
t
e
n
t
/
S
o
f
t
w
a
r
e
A
s
A
G
r
a
p
h

#
 
R
E
P
O
_
D
I
R
 
=
 
"
/
c
o
n
t
e
n
t
/
S
o
f
t
w
a
r
e
A
s
A
G
r
a
p
h
"

#
 
%
c
d
 
{
R
E
P
O
_
D
I
R
}

#
 
%
s
e
t
_
e
n
v
 
P
Y
T
H
O
N
P
A
T
H
=
.



## 2. Hardware Verification & Dependencies
Verify GPU availability and install PyTorch Geometric along with the repository package.

In [ ]:
# Inspect GPU hardware
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("Device Count:", torch.cuda.device_count())
else:
    print("⚠️ GPU not detected. Please enable GPU in Runtime > Change runtime type.")

In [ ]:
# Install PyTorch Geometric, pytest, SimPy, and SaaG core package
!pip install -q torch-geometric pytest simpy tabulate matplotlib scipy
!pip install -q -e .
print("✓ Dependencies installed successfully.")

## 3. Block 0: Pre-Flight Integrity & Stage Contract Audit (Go/No-Go Gate)
Runs the pre-flight verification gate to audit:
1. **Stage & Simulator Contracts** (`tests/test_groundtruth_contract.py`): Verifies the strict decoupling of `FaultInjector` (Predict & Validate CLI), `ValidationService` (`FailureSimulator`), `MessageFlowSimulator` (offline convergent validity probe), and the Input–Label Independence Guarantee of the Explanation Layer (zero simulation runtime access).
2. **QoS Pipeline Integrity** (`tests/test_qos_pipeline_audit.py`): Verifies 16-dim QoS edge feature encoding, mutation gradients, and typed relation projections.
3. **Baseline Formulations** (`tests/test_baselines.py`): Audits capacity parity and parameter counts across structural and homogeneous baselines.
4. **Predictors & Dual-Engine Protocol** (`tests/test_structural_predictors.py`): Audits flow projection derivation (`derive_flow_projection`), `TopoPredictor`, `TopoQoSPredictor`, `DualEnginePredictor`, and ListMLE ranking temperature scaling in `CriticalityLoss`.

In [ ]:
!pytest tests/test_qos_pipeline_audit.py tests/test_baselines.py tests/test_groundtruth_contract.py tests/test_structural_predictors.py -v --tb=short -q


## 4. Smoke-Test (Fast Sanity Check)
Runs a lightweight training sweep on a **3-scenario subset** (`atm_system`, `av_system`, `iot_smart_city_system`) — deliberately smaller than the paper's 7-scenario Table 6 set, just to verify end-to-end execution before running the full 300-epoch matrix. Takes ~1–2 minutes on GPU.

In [ ]:
!python reproduce/main_table.py \
    --scenarios atm_system av_system iot_smart_city_system \
    --seeds 42 123 \
    --epochs 50 \
    --device auto \
    --output results/smoke_main_table.json \
    --resume

## 5. Experiment 1: In-Distribution Model Training & Validation (JSS Table 5 & Table 6)
- **Predict Stage**: Trains 6 model variants across 5 random seeds using ground-truth supervised labels $I^*(v)$ computed by `FaultInjector`. The combined objective optimizes composite regression MSE, dimension-masked reliability, temperature-scaled ListMLE ranking loss ($\tau = 1.0$), pairwise margin ranking ($\gamma = 0.05$), and edge criticality.
- **Validate Stage**: Evaluates trained models against the validation benchmark requiring Spearman rank correlation $\rho \ge 0.70$ and evaluating Top-$K$ critical identification ($F_1@K$, Precision@K, Recall@K, NDCG@10).

Evaluates across the full 12 scenarios:
`atm_system`, `av_system`, `iot_smart_city_system`, `financial_trading_system`, `healthcare_system`, `hub_and_spoke_system`, `microservices_system`, `enterprise_system`, `telecom_ran_system`, `industrial_scada_system`, `realtime_gaming_system`, `logistics_fleet_system`.

- **Structural Baselines**: `Topo` (unweighted flow betweenness + articulation), `Topo-QoS` (QoS-weighted betweenness on flow projection $G_{\text{flow}}$)
- **Homogeneous GNNs**: `GAT`, `GAT-QoS` (scalar QoS)
- **Heterogeneous GNNs**: `HGT` (masked QoS ablation), `HGT-QoS` (proposed model with 16-D continuous-categorical QoS edge features)

*Resilience*: Uses `--resume` so if the Colab session disconnects, re-running skips completed runs.

In [ ]:
# Train full in-distribution matrix — all 12 scenarios (see note above)
!python reproduce/main_table.py \
    --scenarios atm_system av_system iot_smart_city_system financial_trading_system \
                healthcare_system hub_and_spoke_system microservices_system enterprise_system \
                telecom_ran_system industrial_scada_system realtime_gaming_system logistics_fleet_system \
    --output results/main_table.json \
    --seeds 42 123 456 789 2024 \
    --epochs 300 \
    --device auto \
    --resume

In [ ]:
# Render LaTeX and Markdown tables (results/table3_main_results.tex & .md)
!python reproduce/render_table.py \
    --table3 results/main_table.json \
    --output-dir results

# Display rendered markdown table inline
from IPython.display import display, Markdown
if os.path.exists("results/table3_main_results.md"):
    with open("results/table3_main_results.md") as f:
        display(Markdown(f.read()))

## 6. Experiment 2: Inductive Cross-Domain Generalization (LOSO, JSS Table 7 & Table 8)
Evaluates cross-domain inductive generalization via Leave-One-Scenario-Out (LOSO) cross-validation across all 12 domains. Models are trained on 11 domains and evaluated zero-shot on the held-out 12th domain against `FaultInjector` $I^*(v)$. Computes paired Wilcoxon signed-rank tests for statistical significance (JSS Table 7 headline results & Table 8 architectural control arms).

Contrasts learned relational prediction (`HGT-QoS`) against training-free topological centrality (`Topo-QoS` reaches zero-shot $\rho = 0.568$), proving that both paradigms provide strong, complementary signals and justifying their concurrent deployment under the Dual-Engine Consensus Protocol.

In [ ]:
# ── Tables 8 & 9: LOSO re-baseline, all arms in ONE invocation ──────────────
# Every arm must be trained in this single call. loso_significance.py's
# Wilcoxon tests are PAIRED over folds, so pairing hgl_qos from one artifact
# against a control arm from another -- possibly trained against a different
# cache -- is not a paired test.
#
# No --resume. _staleness() compares mtimes against the cache only; it cannot
# see that a previous run was on CPU, so it would happily reuse CPU results in
# a GPU re-baseline. Leftover checkpoints are worse: GNNService resumes from
# whatever it finds, and the driver's own docstring records a dirty workspace
# finishing in 3.2 s at rho = -0.576 where a clean one trained 322 s at +0.594.
!rm -rf output/gnn_checkpoints output/loso output/kfold

!python reproduce/loso_all_variants.py \
    --cache-dir output/loso_cache \
    --variants topo_baseline topo_qos topology_rm gl gl_qos hgl hgl_qos \
               gl_full_cap gl_full_qos_cap gl_full_qos16_cap hgl_qos_uni \
    --epochs 300 \
    --device cuda \
    --output results/loso_all_variants_v5.json

# Sanity check before trusting anything: a fold that "trained" in single-digit
# seconds resumed a stale checkpoint instead of fitting.
import json
folds = json.load(open("results/loso_all_variants_v5.json"))["per_variant_results"]
for variant, payload in folds.items():
    runtimes = [f.get("runtime_s") for f in payload["folds"] if f.get("runtime_s")]
    if runtimes and min(runtimes) < 10:
        print(f"⚠️  {variant}: fastest fold {min(runtimes):.1f}s — suspect a stale checkpoint")
    else:
        print(f"✓ {variant}: {len(payload['folds'])} folds")

# Table 8/9 (manuscript columns only) and the Section 7.2.1 controls table
!python reproduce/render_table.py \
    --table4 results/loso_all_variants_v5.json \
    --table-controls results/loso_all_variants_v5.json \
    --output-dir results

# Significance: pre-registered contrasts, plus the post-hoc RQ2 controls
# Holm-corrected within their own family (PREREGISTRATION.md, Amendment 2).
!python reproduce/loso_significance.py \
    --input results/loso_all_variants_v5.json \
    --output results/loso_significance_v5.json


In [ ]:
# Display summary of LOSO significance
import json
if os.path.exists("results/loso_significance.json"):
    with open("results/loso_significance.json") as f:
        sig = json.load(f)
    print("=== LOSO Wilcoxon Significance Results ===")
    print(json.dumps(sig, indent=2))

## 7. Experiment 3: In-Domain Per-Domain K-Fold Evaluation (JSS Table 9)
Trains 5 variants across $k$ folds per domain scenario, verifying within-domain architectural stability and robustness. Like LOSO, `kfold_all_variants.py` discovers scenarios from `--cache-dir output/loso_cache` rather than a `--scenarios` flag.

In [ ]:
# Optional: run per-domain k-fold cross-validation
!python reproduce/kfold_all_variants.py \
    --cache-dir output/loso_cache \
    --epochs 300 \
    --device auto \
    --output results/kfold_all_variants.json \
    --resume

!python reproduce/render_table.py \
    --table-kfold results/kfold_all_variants.json \
    --output-dir results

## 8. Experiment 4: Real-World Zero-Shot Transfer (JSS Table 11)
Evaluates zero-shot transfer capability of HGT-QoS (trained on synthetic benchmark graphs) and structural baselines when deployed onto five open-source real-world systems: Autoware.Universe (ROS 2), Cloud Microservices, TrainTicket, Home Assistant, and EdgeX Foundry. Models are scored against $I^*(v)$ computed by `FaultInjector`. None of the real-world graphs contribute training gradients or are used for checkpoint selection. Requires `output/realworld_cache` (pulled from Drive in step 1).

*Baseline Enhancement (JSS Section 8.4)*: The framework now natively derives the application flow projection $G_{\text{flow}}$ directly from dependency paths without requiring cached projection files, enabling seamless zero-shot evaluation of both structural baselines (`Topo` and `Topo-QoS`) on arbitrary real-world graphs.

In [ ]:
# Zero-shot transfer of HGT-QoS to five real-world systems
!python reproduce/realworld_zeroshot.py --device auto

## 9. Operational Protocol: Dual-Engine Consensus Gate & Architectural Triage (JSS Section 8.1)
To operationalize JSS Section 8.1's recommendation, SaG provides a **Dual-Engine Consensus Protocol** in `saag predict --dual` running `HGT-QoS` (or GNN/RM) and `Topo-QoS` concurrently over candidate manifests. Because both engines execute in seconds without live infrastructure, the protocol partitions components into two operational sets:

1. **Consensus Critical Set ($S_{\text{consensus}}$)**:
   $$S_{\text{consensus}} = \text{Top}_K(\text{HGT-QoS}) \cap \text{Top}_K(\text{Topo-QoS})$$
   Components flagged by both engines ($K = \lceil 0.2 |V_{\text{app}}| \rceil$) represent high-confidence architectural risks prioritized for mandatory verification of circuit breakers and redundancy policies.
2. **Rank Divergence Escalation ($S_{\text{diverge}}$)**:
   $$S_{\text{diverge}} = \big\{ v \in V_{\text{app}} \;\big|\; |\text{rank}_{\text{HGT-QoS}}(v) - \text{rank}_{\text{Topo-QoS}}(v)| > \Delta_{\text{thresh}} \big\}$$
   Components with substantial rank divergence ($> \Delta_{\text{thresh}}$, default $0.25 |V_{\text{app}}|$) are escalated to human architectural review to investigate edge-case topologies where structural betweenness on $G_{\text{flow}}$ diverges from typed relational attention under declared QoS contracts.

In [ ]:
# Demonstrate Dual-Engine Consensus Protocol programmatically via Python API
import json
from saag.prediction.structural_predictor import DualEnginePredictor, TopoPredictor, TopoQoSPredictor

with open("data/scenarios/av_system.json") as f:
    av_topo = json.load(f)

# Initialize predictors
topo = TopoPredictor()
topo_qos = TopoQoSPredictor()
topo_scores = topo.predict(av_topo)
topo_qos_scores = topo_qos.predict(av_topo)

# Run Dual-Engine consensus analysis (comparing unweighted Topo vs QoS-weighted Topo-QoS)
dual_engine = DualEnginePredictor(topo_predictor=topo_qos, divergence_threshold=3)
dual_res = dual_engine.evaluate_dual(
    gnn_scores=topo_scores,
    graph_or_flow=av_topo,
    k=4
)

print("=== Dual-Engine Consensus Protocol Summary (AV System) ===")
print(f"• Evaluated Components: {len(dual_res.gnn_scores)}")
print(f"• Top-K Threshold (K=4):")
print(f"  - Consensus Critical Set S_consensus ({len(dual_res.consensus_top_k)} nodes): {dual_res.consensus_top_k}")
print(f"  - Divergence Escalation Set S_diverge ({len(dual_res.divergence_escalations)} nodes): {dual_res.divergence_escalations}")
if dual_res.divergence_escalations:
    print("\nEscalation Details:")
    for comp in dual_res.divergence_escalations:
        print(f"  [Escalate] {comp}: Model1 Rank {dual_res.gnn_ranks.get(comp)} vs Topo-QoS Rank {dual_res.topo_ranks.get(comp)} (Δ = {dual_res.rank_divergences.get(comp)})")


## 10. Experiment 5: Inter-Oracle Convergent Validity (JSS Table 13)
- **Convergent Validity Analysis**: Assesses construct and convergent validity across the three simulation oracles:
  - $I^*(v)$ (`FaultInjector`): High-throughput topological subscriber feed-loss cascade oracle used for GNN supervision.
  - $I_{\text{comp}}(v)$ (`FailureSimulator`): Multi-criteria structural cascade composite used for validation gates (`ValidationService`) and counterfactual edit verification (`EditVerifier`).
  - $I_{\text{dyn}}(v)$ (`MessageFlowSimulator`): High-fidelity discrete-event runtime message delivery degradation simulator ($\rho_{\text{util}} = 0.65$, `qos_mode=full`).

High rank correlation ($\rho, \tau$) between the behavioral runtime oracle ($I_{\text{dyn}}$) and structural oracles ($I^*$) demonstrates that topological risk predictions correlate significantly with runtime discrete-event message loss without requiring expensive live traffic simulation in pre-deployment CI/CD.

In [ ]:
# Experiment 5: Run convergent validity analysis across simulation oracles (JSS Table 13)
!python reproduce/convergent_validity.py \
    --scenarios atm_system av_system iot_smart_city_system financial_trading_system \
    --qos-mode full \
    --max-candidates 30 \
    --output results/convergent_validity.json

# Display convergent validity results inline
import json
from pathlib import Path

res_path = Path("results/convergent_validity.json")
if res_path.exists():
    with open(res_path) as f:
        data = json.load(f)
    print("=== Inter-Oracle Convergent Validity (JSS Table 13) ===")
    pairs = data.get("pairwise_agreement", {})
    for pair, metrics in pairs.items():
        rho = metrics.get("spearman_rho", "N/A")
        p_val = metrics.get("spearman_p", "N/A")
        tau = metrics.get("kendall_tau", "N/A")
        jacc = metrics.get("jaccard_top20", "N/A")
        if isinstance(rho, (int, float)):
            print(f"• {pair}: Spearman ρ = {rho:.4f} (p={p_val:.2e}), Kendall τ = {tau:.4f}, Top-20% Jaccard = {jacc:.4f}")
        else:
            print(f"• {pair}: {metrics}")

## 11. Experiment 6: Prescriptive Optimization & Anti-Pattern Detection Efficacy (RQ1)
- **Explanatory Layer**: Computes ISO/IEC 25010 quality dimension decomposition (Reliability $IR$, Maintainability $IM$ via `ChangePropagationSimulator`, and Availability $IA$) with strictly zero simulation runtime access, adhering to the **input–label independence guarantee**.
  - Validates anti-pattern detection efficacy (`reproduce/detection_validation.py`) against `FailureSimulator` ground truth $I_{\text{comp}}$.
- **Prescribe Stage (Remediation Verifier)**: Uses `EditVerifier` powered by `FailureSimulator` counterfactual simulation to verify whether candidate architectural refactorings yield net System Risk Index reduction ($\Delta \text{SRI} > 0$) without introducing secondary cascade regressions (`reproduce/run_prescribe_all.py`).

In [ ]:
# Prescribe Stage: Run Remediation Verifier across benchmark scenarios
!python reproduce/run_prescribe_all.py \
    --scenarios av_system.json iot_smart_city_system.json \
    --kappa 1.0 \
    --output results/prescribe_results.json \
    --resume

# Anti-pattern detection efficacy validation against FailureSimulator ground truth (RQ1)
!python reproduce/detection_validation.py \
    --scenarios av_system iot_smart_city_system financial_trading_system \
    --output results/detection_validation.json

# Display prescriptive remediation verification inline
import json
from pathlib import Path

if Path("results/prescribe_results.json").exists():
    with open("results/prescribe_results.json") as f:
        prescribe_data = json.load(f)
    print("=== Prescribe Stage: Remediation Verification Summary ===")
    for sc, res in prescribe_data.items():
        base_sri = res.get("baseline_sri", "N/A")
        opt_sri = res.get("optimized_sri", "N/A")
        delta = res.get("delta_sri", "N/A")
        ops = res.get("applied_operators", 0)
        print(f"• {sc}: Baseline SRI = {base_sri}, Optimized SRI = {opt_sri}, Δ = {delta} ({ops} operators applied)")

## 12. Case Study & Figures (Attention Subgraphs & JSS Figures 3–5)
Generates the publication figures:
- **Figure S2 / Figure 5**: ATM Case Study HGT Attention Subgraph.
- **Figure 3**: Results-at-a-glance (LOSO Spearman $\rho$, F1@K, and inter-oracle agreement across simulators).
- **Figure 4**: Stratified per-node-type $\rho$.


In [ ]:
# Extract attention weights for ATM scenario and render subgraph
!python reproduce/extract_attention.py \
    --scenario atm_system \
    --device auto \
    --output-dir output/atm_case_study

!python reproduce/render_attention_subgraph.py \
    --input output/atm_case_study/attention_weights.json \
    --output output/atm_case_study/attention_subgraph

# Render results-at-a-glance figure
!python reproduce/render_results_figure.py
!python reproduce/render_stratified_figure.py --source auto --output results/figure4_stratified_rho

In [ ]:
# Display rendered figures
from IPython.display import Image, display
from pathlib import Path

figures = [
    Path("output/atm_case_study/attention_subgraph.png"),
    Path("docs/research/jss/latex/figures/Figure_3.png"),
    Path("results/figure4_stratified_rho.png")
]

for fig in figures:
    if fig.exists():
        print(f"\nFigure: {fig}")
        display(Image(filename=str(fig)))

## 13. Pre-Publication Reconciliation Gate (Zero-Drift Verification)
Runs `reproduce/reconcile_manuscript.py` to assert that all experimental results, tables, and metrics generated in this notebook match the published JSS manuscript tables across all 181 reported figures with 0 mismatches before committing or distributing results.


In [ ]:
# Reconcile generated experimental artifacts against published JSS manuscript tables
!python reproduce/reconcile_manuscript.py


## 14. Backup Checkpoints & Results to Google Drive
Persists trained model checkpoints (`output/gnn_checkpoints/`), evaluation tables, figures, and stage artifacts (`results/`) to Google Drive.


In [ ]:
from datetime import datetime

backup_dir = f"/content/drive/MyDrive/SaG_JSS_Results_{datetime.now().strftime('%Y%m%d_%H%M')}"
os.makedirs(backup_dir, exist_ok=True)

!cp -r results {backup_dir}/
if os.path.exists("output/gnn_checkpoints"):
    !cp -r output/gnn_checkpoints {backup_dir}/
if os.path.exists("output/atm_case_study"):
    !cp -r output/atm_case_study {backup_dir}/

print(f"✓ Experimental artifacts backed up to: {backup_dir}")